In [36]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

URL = "https://finance.yahoo.com/markets/crypto/most-active/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def parse_human_readable(value):
    if not value:
        return None
    val = value.replace(',', '')
    last_char = val[-1]
    val_no_last_char = val[:-1]
    match(last_char):
        case 'K':
            return float(val_no_last_char) * 1000
        case 'M':
            return float(val_no_last_char) * 1_000_000
        case 'B':
            return float(val_no_last_char) * 1_000_000_000
        case 'T':
            return float(val_no_last_char) * 1_000_000_000_000
        case _:
            return float(val)


def crypto_volumes():
    response = requests.get(URL, headers=HEADERS)
    #print(response)
    html = response.text
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", attrs={"class": "yf-19xszjc"})
    #print(table)
    rows = table.find('tbody').find_all('tr')
    #print(rows)
    data = []
    for row in rows:
        #print(row)
        #print('---------------------')
        cells = row.find_all('td')

        symbol = cells[0].text.strip()
        name = cells[1].text.strip()
        price = cells[4].text.strip()
        market_cap = cells[6].text.strip()
        volume = cells[8].text.strip()
        record = {
            'symbol': symbol,
            'name': name,
            'price': parse_human_readable(price),
            'market_cap': parse_human_readable(market_cap),
            'volume': parse_human_readable(volume)
        }
        #print(record)
        data.append(record)
    df = pd.DataFrame(data)
    #df = df.set_index('symbol')
    return df

def save_crypto_volumes(df):
    print(df)
    df.to_csv('data/crypto_volumes.csv', index=False)
    df.to_excel('data/crypto_volumes.xlsx', index=False)
    df.to_json('data/crypto_volumes.json', orient='records', indent=4)
    #df.to_json('data/crypto_volumes.json', index=False)

In [46]:
import plotly.express as px

def human_readable(value):
    return f'{value / 1_000_000_000}B'

def plot_volume(df):
    df['market_capitalization'] = df['market_cap'].apply(lambda x: human_readable(x))
    df_sorted = df.sort_values(by=['volume'], ascending=False)
    df_sorted_big = df_sorted[df_sorted['volume'] > 1_000_000_000]

    fig = px.bar(df_sorted_big, x='symbol', y='volume', title='Crypto Volumes - Yahoo Finance',
                 hover_data=['price', 'market_capitalization', 'volume'])
    fig.show()

In [45]:
def x2(x):
    return x*2

print(x2(1000))

x_2 = lambda x: x*2
print(x_2(1000))

2000
2000


In [47]:
df = crypto_volumes()
save_crypto_volumes(df)
plot_volume(df)

<Response [200]>
---------------------
{'symbol': 'USDT-USD', 'name': 'Tether USDt USD', 'price': 0.0, 'market_cap': 176798000000.0, 'volume': 55690000000.0}
---------------------
{'symbol': 'BTC-USD', 'name': 'Bitcoin USD', 'price': -1714.96, 'market_cap': 1489000000000.0, 'volume': 25167000000.0}
---------------------
{'symbol': 'ETH-USD', 'name': 'Ethereum USD', 'price': -134.81, 'market_cap': 256570000000.0, 'volume': 14154000000.0}
---------------------
{'symbol': 'USDC-USD', 'name': 'USDC USD', 'price': -0.0, 'market_cap': 70239000000.0, 'volume': 9834000000.0}
---------------------
{'symbol': 'SOL-USD', 'name': 'Solana USD', 'price': -3.1, 'market_cap': 50842000000.0, 'volume': 2729000000.0}
---------------------
{'symbol': 'XRP-USD', 'name': 'XRP USD', 'price': -0.07, 'market_cap': 69182000000.0, 'volume': 2664000000.0}
---------------------
{'symbol': 'BNB-USD', 'name': 'BNB USD', 'price': -17.15, 'market_cap': 89384000000.0, 'volume': 1531000000.0}
---------------------
{'sym